Set up the VIIRS NDVI and EVI data to be processed using Google Earth Engine

Author: Tayler LaSharr;
Date: 10 Sept 2023


In [ ]:
## first we will read in necessary python libraries and intialize the GEE module
## you will have to authenticate GEE on your machine to use this code

import ee
import matplotlib.pyplot as plt
from datetime import datetime
import geopandas as gpd
import matplotlib.pyplot as plt

## if you need to authenticate GEE use this code
## ee.Authenticate('your-GEE-project')

## initialize the Earth Engine module
ee.Initialize()

In [ ]:
## load in a shapefile or area of interest
## this is an assest that is already loaded into GEE

md_cooridor = ee.FeatureCollection('projects/ee-tlasharr/assets/UT_Paunsaugunt')

## load in the VIIRS data and the MODIS snow data
viirs_ndvi = ee.ImageCollection('NASA/VIIRS/002/VNP13A1') \
                .filterDate('2018-01-01', '2023-12-31') \
                .select('NDVI', 'Pixel_Quality)

viirs_evi = ee.ImageCollection('NASA/VIIRS/002/VNP13A1') \
                .filterDate('2018-01-01', '2023-12-31') \
                .select('EVI', 'Pixel_Quality)

modis_snow = ee.ImageCollection('MODIS/006/MOD10A1') \
                .filterDate('2018-01-01', '2023-12-31') \
                .select('NDSI_Snow_Cover')

## first see the projection of the shapefile
coordidor_projection = md_cooridor.first().geometry().projection()
print('Projection of the cooridor shapefile:', coordidor_projection.getInfo())

## now we can reproject the shapefile
viirs_projection = viirs_ndvi.first().projection() ## this is the projection of the first image of the modis ndvi data
target_crs = viirs_projection.crs() ## this is the crs from the modis.projection object
target_crs_scale = viirs_projection.nominalScale() ## this is the scale (confused on what this is need to figure this out)

## this is a function to reproject it
def reproject_feature(feature):
    return feature.setGeometry(feature.geometry().transform(target_crs, target_crs_scale))

## this is running the function, so now "reprojected_roi" is the new object we care about
reprojected_roi = md_cooridor.map(reproject_feature)

## then plot it
reprojected_roi.plot()
reprojected_roi.plot(column='attribute', cmap='OrRd', legend=True)
plt.show()               

In [ ]:
## next, we can clip the data to the cooridor that we are working with

## clip VIIRS NDVI and EVI datasets
clipped_viirs_ndvi = viirs_ndvi.map(lambda image: image.clip(reprojected_roi))
clipped_viirs_evi = viirs_evi.map(lambda image: image.clip(reprojected_roi))

Everything above this is just getting the raw data clipped and ready to go for processing. Next, we are going to actually process the data. We have altered the methods slightly from Bischof's 2012 methods to help us fit curves in arid environments.

We are going to try to develop some methods that can be used for all of the data, so that we don't have to run seperate methods for temperate and arid landscapes.

In [ ]:
## first thing we will do is 

In [ ]:
## now we can run the smoothing process on the data following Bischof et al. 2012's methods

## define a function to use a moving median filter to reduce noise
def apply_median_filter(image):
    
    ## Apply focal median filter with a radius of 3 pixels
    smoothed_image = image.focal_median(radius=3, units='pixels')
    
    ## Retain the 'system:time_start' property
    return smoothed_image.set('system:time_start', image.get('system:time_start'))


## now use the function on the NDVI image collection
smoothed_viirs_ndvi = clipped_viirs_ndvi.map(apply_median_filter)

In [ ]:
## now we will fix snow in the NDVI data. For now, I am going to shift slightly from 
## the Bischof methods because I don't want to overestimate green-up for desert systems

## to do this, first we will define a function to mask snow cover,
## this is a step to help us understand how snow plays into the curves
## especially important for desert systems, right now I set it to mask 
## cells with more than 35% snow cover

def apply_snow_mask(image, snow_image):
    snow_mask = ee.Algorithms.If(
        snow_image,
        snow_image.lte(35),  ## if the snow cover is less than or equal to 35%
        ee.Image(1)          ## default mask (all values included if no corresponding snow image)
    )
    return image.updateMask(snow_mask)

## now mask the NDVI data with more than 35% snow cover
viirs_ndvi_without_snow = clipped_viirs_ndvi.map(
    lambda image: apply_snow_mask(image, modis_snow.filterDate(image.date(), image.date().advance(1, 'day')).first())
)

## now mask the EVI data with more than 35% snow cover
viirs_evi_without_snow = clipped_viirs_evi.map(
    lambda image: apply_snow_mask(image, modis_snow.filterDate(image.date(), image.date().advance(1, 'day')).first())
) 